# ParkMap — YOLO11s training and evaluation (960 px)

Notebook สำหรับ Google Colab ใช้ทดลอง YOLO11s; ไม่ได้เปลี่ยน weights ที่ Vision API ใช้อัตโนมัติ
ภาพต้นฉบับยังเก็บที่ resolution เดิม โดย YOLO resize ภายในตาม `IMGSZ=960`.
เปลี่ยน `BASE_WEIGHTS` เป็น YOLO11n ได้เมื่อต้องการทดลองรุ่นเดียวกับ baseline เดิม แต่ต้องรายงานเป็นคนละ experiment.

ก่อนฝึก: แยก **train / val / test ตามรอบถ่าย** และล็อก test ก่อนเริ่มเลือกโมเดล
ห้ามใช้ test เป็น val และห้ามเลือกโมเดล/threshold จากผล test ซ้ำ ๆ.
ถ้ามีเพียง 5 ภาพ train กับ 2 ภาพที่ใช้เป็นทั้ง val/test ให้เตรียม split อิสระก่อน;
ผลเดิมถือเป็นการทดลองเบื้องต้น ไม่ใช่หลักฐานความแม่นยำในลานจริง.

รูป labels weights และ outputs มีข้อมูลส่วนตัว: เก็บใน Drive ของทีม ไม่ใส่ Git.
ภาพตัวอย่างปิดไว้โดยค่าเริ่มต้น; ก่อน commit ให้ Clear all outputs และ execution counts.
คำสั่งเทรนและประเมิน test ปิดไว้ ต้องเปิดเองในหัวข้อที่เกี่ยวข้อง.

## 1. Environment
เลือก GPU runtime ใน Colab ถ้าต้องการฝึกด้วย GPU. Pin Ultralytics ให้ตรงกับโปรเจกต์;
Torch/CUDA ใช้ environment ของ Colab และบันทึกเวอร์ชันจริงในรายงานแต่ละ run.
หากเพิ่งติดตั้ง dependency แล้ว Colab ขอ restart runtime ให้ restart ก่อนรันต่อ.

In [ ]:
%pip install -q ultralytics==8.4.166

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration — แก้ค่าที่นี่จุดเดียว
ตั้งค่าประเมินและ `EVAL_CHECKPOINT` ในหัวข้อ 6 หลังเลือกโมเดลจาก val แล้ว.
test metrics จากหลาย checkpoint ใช้เป็นการสำรวจเท่านั้น ไม่ใช่ untouched-test evidence.

In [ ]:
import hashlib
import importlib.metadata
import json
import math
import platform
from datetime import datetime, timezone
from pathlib import Path

import torch
import yaml
from PIL import Image, ImageOps
from ultralytics import YOLO

PROJECT = Path('/content/drive/MyDrive/spu/year 4/Project')
DATASET = PROJECT / 'dataset/original_images'
ARTIFACT_ROOT = PROJECT / 'runs/parkmap'
BASE_WEIGHTS = 'yolo11s.pt'
IMGSZ = 960
CONF = 0.25
NMS_IOU = 0.50
MATCH_IOU = 0.50
MAX_DET = 300
MOTORCYCLE_CLASS_ID = 0
EPOCHS = 150
SEED = 42
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
BATCH = 4 if torch.cuda.is_available() else 1
RUN_NAME = f'{Path(BASE_WEIGHTS).stem}_spu_{IMGSZ}_b{BATCH}_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}'
RUN_TRAINING = False
SHOW_PRIVATE_IMAGES = False
TRAINED_BEST = None
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
ENVIRONMENT = {'python': platform.python_version(), 'torch': torch.__version__,
               'cuda': torch.version.cuda, 'device': str(DEVICE),
               'packages': {name: importlib.metadata.version(name)
                            for name in ('ultralytics', 'numpy', 'Pillow', 'PyYAML')}}
print(json.dumps(ENVIRONMENT, indent=2))

## 3. Dataset validation — หยุดเมื่อข้อมูลผิด

ต้องมี `train/images`, `train/labels`, `val/images`, `val/labels`, `test/images`, `test/labels`.
ทุกภาพมี label ชื่อเดียวกัน; ภาพที่ตรวจแล้วว่าไม่มีรถใช้ไฟล์ label ว่างอย่างชัดเจน.
ตรวจ class 0, จำนวนค่า, finite values, กล่องไม่เลยขอบภาพ, ภาพเสีย, EXIF orientation,
stem ซ้ำ และไฟล์ภาพซ้ำข้าม split ด้วย SHA-256.

เพิ่ม `DATASET/sessions.json` เป็น object จับคู่ **ชื่อภาพไม่มีนามสกุล → รอบถ่ายจริง** เช่น
`{"frame_001": "2026-10-28-morning", "frame_100": "2026-10-29-afternoon"}`.
ต้องใส่ครบทุกภาพ และ session เดียวกันห้ามอยู่คนละ split.
ห้ามตั้ง session คนละชื่อเพียงเพื่อหลบการตรวจ; การสุ่มแยกเฟรมจากการถ่ายครั้งเดียวไม่ใช่ split อิสระ.
hash ตรวจได้เฉพาะไฟล์เหมือนกัน ส่วน near-duplicates ต้องใช้ข้อมูลรอบถ่ายและตรวจด้วยคนด้วย.
Notebook อ่านและตรวจเท่านั้น ไม่ย้ายภาพหรือแบ่ง dataset ให้เอง.

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def get_images(folder):
    folder = Path(folder)
    if not folder.is_dir():
        raise ValueError(f'Missing image directory: {folder}')
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def read_yolo_labels(path):
    path = Path(path)
    if not path.is_file():
        raise ValueError(f'Missing label: {path.name}; use an empty file only for a reviewed background image')
    boxes = []
    for number, line in enumerate(path.read_text(encoding='utf-8-sig').splitlines(), 1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5 or parts[0] != '0':
            raise ValueError(f'{path.name}:{number}: expected integer class 0 and four coordinates')
        try:
            x, y, w, h = map(float, parts[1:])
        except ValueError as error:
            raise ValueError(f'{path.name}:{number}: invalid number') from error
        if not all(math.isfinite(v) for v in (x, y, w, h)) or w <= 0 or h <= 0:
            raise ValueError(f'{path.name}:{number}: box must be finite with positive width/height')
        box = [x - w / 2, y - h / 2, x + w / 2, y + h / 2]
        if not all(-1e-6 <= v <= 1 + 1e-6 for v in box):
            raise ValueError(f'{path.name}:{number}: box extends outside the image')
        boxes.append(box)
    return boxes


def validate_dataset(root):
    root = Path(root)
    manifest = root / 'sessions.json'
    if not manifest.is_file():
        raise ValueError('Create dataset/sessions.json mapping every image stem to its real capture session first')
    sessions = json.loads(manifest.read_text(encoding='utf-8'))
    if not isinstance(sessions, dict):
        raise ValueError('sessions.json must be an object mapping image stems to capture session names')
    used_stems, used_hashes, used_sessions = {}, {}, {}
    report = {}
    for split in ('train', 'val', 'test'):
        images = get_images(root / split / 'images')
        label_dir = root / split / 'labels'
        if not images or not label_dir.is_dir():
            raise ValueError(f'{split}: requires non-empty images and a labels directory; never substitute test for val')
        stems = [p.stem for p in images]
        if len(set(stems)) != len(stems):
            raise ValueError(f'{split}: multiple image extensions share the same label stem')
        label_stems = {p.stem for p in label_dir.glob('*.txt')}
        if label_stems != set(stems):
            raise ValueError(f'{split}: image/label mismatch: missing={sorted(set(stems) - label_stems)}, orphan={sorted(label_stems - set(stems))}')
        boxes, sizes, split_sessions = 0, set(), set()
        digest = hashlib.sha256()
        for image in images:
            label = label_dir / f'{image.stem}.txt'
            if image.stem in used_stems:
                raise ValueError(f'Image stem crosses splits: {image.stem}')
            used_stems[image.stem] = split
            session = sessions.get(image.stem)
            if not isinstance(session, str) or not session.strip():
                raise ValueError(f'Missing capture session for {image.stem}')
            if session in used_sessions and used_sessions[session] != split:
                raise ValueError(f'Capture session {session} crosses splits')
            used_sessions[session] = split
            split_sessions.add(session)
            file_hash = sha256(image)
            if file_hash in used_hashes and used_hashes[file_hash] != split:
                raise ValueError('Identical image bytes cross dataset splits')
            used_hashes[file_hash] = split
            with Image.open(image) as decoded:
                if decoded.getexif().get(274, 1) != 1:
                    raise ValueError(f'{image.name}: normalize EXIF orientation and review labels first')
                sizes.add(decoded.size)
                decoded.load()
            boxes += len(read_yolo_labels(label))
            digest.update(json.dumps([image.name, file_hash, sha256(label), session]).encode('utf-8'))
        report[split] = {'images': len(images), 'boxes': boxes, 'resolutions': sorted(sizes),
                         'sessions': sorted(split_sessions), 'sha256': digest.hexdigest()}
    return report


DATASET_REPORT = validate_dataset(DATASET)
print(json.dumps(DATASET_REPORT, indent=2))

## 4. Optional private ground-truth preview
เปิด `SHOW_PRIVATE_IMAGES` เฉพาะขณะตรวจใน Drive ส่วนตัว. ตรวจด้วยตาหลังแปลง HEIC/JPEG;
ชื่อไฟล์และกล่องตรงกันไม่ได้พิสูจน์ว่า labels อยู่บนรถจริง. Clear outputs ก่อนนำ notebook เข้า Git.

In [ ]:
if SHOW_PRIVATE_IMAGES:
    import random
    import matplotlib.pyplot as plt
    from PIL import ImageDraw
    for split in ('train', 'val', 'test'):
        images = get_images(DATASET / split / 'images')
        random.Random(SEED).shuffle(images)
        for path in images[:2]:
            with Image.open(path) as original:
                preview = ImageOps.exif_transpose(original).convert('RGB')
            draw = ImageDraw.Draw(preview)
            for x1, y1, x2, y2 in read_yolo_labels(DATASET / split / 'labels' / f'{path.stem}.txt'):
                draw.rectangle((x1 * preview.width, y1 * preview.height, x2 * preview.width, y2 * preview.height),
                               outline='lime', width=max(2, preview.width // 1500))
            plt.figure(figsize=(12, 8))
            plt.imshow(preview)
            plt.title(f'{split}: {path.name}')
            plt.axis('off')
            plt.show()
            plt.close()
else:
    print('Private image preview disabled')

## 5. Train — YOLO11s, 960 px
เปิด `RUN_TRAINING=True` หลังตรวจ dataset และ GT แล้ว. Auto-select CUDA เมื่อมี; CPU ใช้ batch=1.
สร้าง data.yaml แยกใน artifact directory ไม่เขียนทับ data.yaml ของ dataset.
แต่ละ run ชื่อไม่ซ้ำและไม่ overwrite; best.pt อ้างจาก trainer ของรอบนี้โดยตรง.
การกดรัน training cell ซ้ำจะถูกปฏิเสธ ให้รัน Configuration ใหม่เพื่อสร้างชื่อรอบใหม่.

In [ ]:
if RUN_TRAINING:
    if validate_dataset(DATASET) != DATASET_REPORT:
        raise ValueError('Dataset changed since validation; review it again before training')
    if (ARTIFACT_ROOT / RUN_NAME).exists():
        raise ValueError('Run already exists; rerun Configuration to create a new experiment')
    ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    data_yaml = ARTIFACT_ROOT / f'{RUN_NAME}.data.yaml'
    data_yaml.write_text(yaml.safe_dump({'path': str(DATASET.resolve()),
        'train': 'train/images', 'val': 'val/images', 'test': 'test/images',
        'names': {0: 'motorcycle'}}, sort_keys=False), encoding='utf-8')
    model = YOLO(BASE_WEIGHTS)
    model.train(data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                workers=2, patience=30, device=DEVICE, degrees=3.0, translate=0.08,
                scale=0.25, fliplr=0.5, mosaic=0.5, close_mosaic=10, seed=SEED,
                deterministic=True, project=str(ARTIFACT_ROOT), name=RUN_NAME, exist_ok=False)
    TRAINED_BEST = Path(model.trainer.best)
    if not TRAINED_BEST.is_file():
        raise FileNotFoundError('Training did not produce best.pt; evaluation is blocked')
    training_record = {'base_weights': BASE_WEIGHTS, 'checkpoint': str(TRAINED_BEST),
        'weights_sha256': sha256(TRAINED_BEST), 'imgsz': IMGSZ, 'batch': BATCH,
        'seed': SEED, 'environment': ENVIRONMENT, 'dataset': DATASET_REPORT}
    (TRAINED_BEST.parent.parent / 'parkmap-training.json').write_text(
        json.dumps(training_record, indent=2), encoding='utf-8')
    print('Training complete. Checkpoint:', TRAINED_BEST)
    print('Checkpoint SHA-256:', training_record['weights_sha256'])
else:
    print('Training disabled; set RUN_TRAINING=True when ready')

## 6. Select checkpoint and evaluation configuration
แก้ switches ใน cell ถัดไปหลังฝึก แล้วรันตั้งแต่หัวข้อนี้ลงไป ไม่ต้องรัน Configuration หรือฝึกซ้ำ.
`EVAL_CHECKPOINT=None` ใช้ best.pt จากการฝึกใน session นี้; ถ้าเริ่ม runtime ใหม่ ให้ระบุ path เดิมชัดเจน.
ใช้ checkpoint จุดเดียวกันทั้ง mAP, per-image comparison และ custom TP/FP/FN.
โมเดลต้องเป็น single-class `motorcycle` ที่ class 0 เหมือน API.
NMS_IOU ตัดกล่องซ้ำตอน inference; MATCH_IOU ใช้จับคู่กับ GT. เป็นคนละขั้นตอนแม้ค่าจะเท่ากัน.
เปรียบเทียบกับเว็บต้องตรงทั้ง SHA-256 ของ weights/ภาพ, imgsz, confidence, NMS, max_det,
เวอร์ชัน library และ preprocessing. จำนวน prediction ไม่ใช่จำนวนรถจริง และไม่กำหนดตายตัวว่าเป็น 86.

In [ ]:
RUN_TEST_EVALUATION = False  # Enable only after selecting the checkpoint using val
RUN_SINGLE_IMAGE_CHECK = False
EVAL_CHECKPOINT = None  # Or Path('/private/experiment/weights/best.pt')
CHECK_IMAGE = None     # Exact uploaded image; no fixed expected prediction count
EVALUATION_ID = f'{RUN_NAME}_eval_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}'
BEST = None
best_model = None
if RUN_TEST_EVALUATION or RUN_SINGLE_IMAGE_CHECK:
    checkpoint = EVAL_CHECKPOINT if EVAL_CHECKPOINT is not None else TRAINED_BEST
    if checkpoint is None or not Path(checkpoint).is_file():
        raise FileNotFoundError('Train in this session or explicitly set EVAL_CHECKPOINT to a reviewed checkpoint')
    BEST = Path(checkpoint)
    best_model = YOLO(str(BEST))
    names = best_model.names
    names = names if isinstance(names, dict) else dict(enumerate(names))
    if set(names) != {0} or str(names[0]).strip().lower() != 'motorcycle':
        raise ValueError('Expected a single-class checkpoint with class 0 = motorcycle')
    print('Checkpoint SHA-256:', sha256(BEST))
else:
    print('Evaluation and single-image comparison disabled')

## 7. Held-out test mAP
เปิด `RUN_TEST_EVALUATION` หลังเลือก checkpoint จาก val เสร็จแล้วเท่านั้น.
Ultralytics mAP กับ custom Precision/Recall ที่ threshold เดียวเป็นคนละ metric; รายงานแยกกัน.
ผลชุดเล็ก ภาพเดิม หรือการทดลองซ้ำบน test ไม่ใช่หลักฐาน accuracy ในลานจริง.

In [ ]:
MAP_REPORT = None
if RUN_TEST_EVALUATION:
    if validate_dataset(DATASET) != DATASET_REPORT:
        raise ValueError('Dataset changed since validation; review splits before evaluation')
    eval_yaml = ARTIFACT_ROOT / f'{EVALUATION_ID}.data.yaml'
    eval_yaml.parent.mkdir(parents=True, exist_ok=True)
    eval_yaml.write_text(yaml.safe_dump({'path': str(DATASET.resolve()),
        'train': 'train/images', 'val': 'val/images', 'test': 'test/images',
        'names': {0: 'motorcycle'}}, sort_keys=False), encoding='utf-8')
    metrics = best_model.val(data=str(eval_yaml), split='test', imgsz=IMGSZ,
                             conf=CONF, iou=NMS_IOU, max_det=MAX_DET, device=DEVICE,
                             project=str(ARTIFACT_ROOT), name=EVALUATION_ID, exist_ok=False)
    MAP_REPORT = {'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map)}
    print(json.dumps(MAP_REPORT, indent=2))

## 8. Greedy confidence-ordered TP / FP / FN
อ่าน labels แบบเดียวกับการตรวจ dataset. Missing label เป็น error; กล่องผิดไม่ถูกข้ามเงียบ ๆ.
เรียง prediction ตาม confidence ก่อนจับคู่ IoU กับ GT ที่ยังไม่ถูกใช้ ตาม protocol ของโปรเจกต์.
กรณีไม่มี GT หรือไม่มี prediction รายงาน metric ที่หารไม่ได้เป็น null แทนการอ้างว่าได้ 0%.
รายงานนี้ประเมิน bbox เท่านั้น ยังไม่ได้วัด Parking ROI misses หรือแยก occlusion ตาม protocol เต็มรูปแบบ.

In [ ]:
def box_iou(a, b):
    intersection = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def evaluate_image(predictions, gt_boxes, match_iou=0.5):
    matched = set()
    tp, fp = 0, 0
    for prediction in sorted(predictions, key=lambda p: p['confidence'], reverse=True):
        choices = [(box_iou(prediction['box'], gt), i) for i, gt in enumerate(gt_boxes) if i not in matched]
        overlap, index = max(choices, default=(0.0, -1), key=lambda pair: pair[0])
        if index >= 0 and overlap >= match_iou:
            matched.add(index)
            tp += 1
        else:
            fp += 1
    return tp, fp, len(gt_boxes) - len(matched)


if RUN_TEST_EVALUATION:
    if best_model is None or BEST is None:
        raise ValueError('Run checkpoint selection before evaluation')
    if validate_dataset(DATASET) != DATASET_REPORT:
        raise ValueError('Dataset changed since validation')
    per_image = []
    for path in get_images(DATASET / 'test/images'):
        gt = read_yolo_labels(DATASET / 'test/labels' / f'{path.stem}.txt')
        result = best_model.predict(source=str(path), imgsz=IMGSZ, conf=CONF,
            iou=NMS_IOU, max_det=MAX_DET, classes=[MOTORCYCLE_CLASS_ID], device=DEVICE, verbose=False)[0]
        predictions = [{'box': [float(v) for v in box.xyxyn[0].cpu().tolist()],
                        'confidence': float(box.conf.item())} for box in result.boxes]
        tp, fp, fn = evaluate_image(predictions, gt, MATCH_IOU)
        per_image.append({'image': path.name, 'gt': len(gt), 'predictions': len(predictions), 'tp': tp, 'fp': fp, 'fn': fn})
    tp, fp, fn = (sum(item[key] for item in per_image) for key in ('tp', 'fp', 'fn'))
    precision = tp / (tp + fp) if tp + fp else None
    recall = tp / (tp + fn) if tp + fn else None
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None
    report = {'weights_sha256': sha256(BEST), 'checkpoint': str(BEST), 'environment': ENVIRONMENT,
        'inference': {'imgsz': IMGSZ, 'confidence': CONF, 'nms_iou': NMS_IOU, 'max_det': MAX_DET, 'class_id': MOTORCYCLE_CLASS_ID},
        'match_iou': MATCH_IOU, 'dataset': DATASET_REPORT, 'map': MAP_REPORT,
        'metrics': {'tp': tp, 'fp': fp, 'fn': fn, 'precision': precision, 'recall': recall, 'f1': f1},
        'per_image': per_image, 'limitation': 'Bounding-box evaluation only; no ROI-miss or occlusion breakdown'}
    report_path = ARTIFACT_ROOT / f'{EVALUATION_ID}.evaluation.json'
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report['metrics'], indent=2))
    print('Evaluation report saved outside Git')

## 9. Optional comparison with one exact web-upload image
กำหนด `CHECK_IMAGE` และเปิด `RUN_SINGLE_IMAGE_CHECK`. ใช้ BEST และ configuration เดียวกับ evaluation.
เปรียบเทียบกับ inference output ของเว็บก่อนกรอง ROI; จำนวนรถหลังกรอง ROI เป็นคนละขั้นตอน.
บันทึก image/weights SHA-256 เพื่อยืนยันว่าใช้ไฟล์เดียวกัน ไม่ใช้ชื่อไฟล์เป็นหลักฐานอย่างเดียว.

In [ ]:
if RUN_SINGLE_IMAGE_CHECK:
    if best_model is None or BEST is None:
        raise ValueError('Run checkpoint selection first')
    if CHECK_IMAGE is None or not Path(CHECK_IMAGE).is_file():
        raise FileNotFoundError('Set CHECK_IMAGE to the exact file uploaded to the web')
    prediction = best_model.predict(source=str(CHECK_IMAGE), imgsz=IMGSZ, conf=CONF,
        iou=NMS_IOU, max_det=MAX_DET, classes=[MOTORCYCLE_CLASS_ID], device=DEVICE, verbose=False)[0]
    print(json.dumps({'weights_sha256': sha256(BEST), 'image_sha256': sha256(CHECK_IMAGE),
                     'predictions': len(prediction.boxes), 'imgsz': IMGSZ,
                     'confidence': CONF, 'nms_iou': NMS_IOU, 'max_det': MAX_DET}, indent=2))

## 10. Before using a checkpoint in ParkMap
เก็บ training/evaluation JSON กับ weights ไว้ใน Drive ของทีม. ตรวจรายงานและชุด test อิสระก่อน
เลือกนำ checkpoint ไปใช้ใน API; notebook ไม่คัดลอกทับ `apps/vision-api/models/best.pt`.
เทียบค่า runtime ใน `apps/vision-api/app/services/cv_service.py` และ `PARKMAP_MODEL_*` ด้วย.
baseline เก่าเป็นคนละ experiment จนกว่าจะยืนยัน weights SHA-256, dataset และ configuration ตรงกัน.
ล้าง outputs/execution counts ก่อน commit เสมอ โดยเฉพาะหลังเปิด private preview.